In [1]:
import pandas as pd
import numpy as np
import os
import time
import pickle
from matplotlib import pyplot as plt

#loc_dir='/Users/daniel.garger/Desktop/Projects/C-PATH/'
#cluster_dir='/mnt/home/icb/daniel.garger/'
#os.chdir(loc_dir)
#data_dir='TTP prediction/data'
os.getcwd()
os.chdir(os.getcwd())
os.getcwd()

'/data/gpfs/projects/punim2121/C-Path/outcome_prediction/outcome_prediction'

In [2]:
xo=pd.read_csv('../../C-Path_data/fullExportDb-1025-Member-CSV/xo.csv',low_memory=False)
da=pd.read_csv('../../C-Path_data/fullExportDb-1025-Member-CSV/da.csv',low_memory=False)
ds=pd.read_csv('../../C-Path_data/fullExportDb-1025-Member-CSV/ds.csv',low_memory=False)
pat_id_df=pd.read_csv('../data/patients_in_analysis.csv.gz',index_col=0)

In [3]:
xo['STUDYID'].unique()

array(['TB-1021', 'TB-1022', 'TB-1030', 'TB-1020'], dtype=object)

## TB-1021: OUTCOME AT END OF TREATMENT, 12, 18 MONTHS
* ### Combined Failure of Bacteriological Cure and Relapse Within One Year of Completion of Therapy as Defined by Culture Using Solid Media (Lowenstein-Jensen - LJ). [ Time Frame: 18 months (within one year of completion of therapy) ]
* ### The primary efficacy outcome was the proportion of patients who had bacteriologically or clinically defined failure or relapse within 18 months after randomization (a composite unfavorable outcome).

In [4]:
## Select approach (per protocol vs. modified intent to treat)
analysis_approach='PER PROTOCOL'

## Subset the xo dataframe
xo_1021=xo[xo['STUDYID']=='TB-1021']
xo_1021=xo_1021.replace({'PER PROTCOL':'PER PROTOCOL'})

## Initialise a dataframe to collect the outcomes to
outcome_columns=['RESULT_AT_END_OF_TREATMENT','RESULT_AT_12_MONTHS','RESULT_LIQUID_MEDIUM_AT_18_MONTHS',
                'UNFAVOURABLE_OUTCOME_CATEGORY_AT_18_MONTHS']
outcome_tb1021=pd.DataFrame(index=xo_1021['USUBJID'].unique(),columns=outcome_columns)

## Extract outcomes at different timepoints
res_end_of_treat=xo_1021[(xo_1021['XOTEST']=='Status') & (xo_1021['XOCAT'].str.contains(analysis_approach,na=False)) &
                            (xo_1021['XOTPT'].str.contains('END OF TREATMENT'))]   
res_12_month=xo_1021[(xo_1021['XOTEST']=='Status') &(xo_1021['XOCAT'].str.contains(analysis_approach,na=False)) &
                            (xo_1021['XOTPT'].str.contains('12'))]

## As per study description the outcomes at 18 Months were determined on solid and liquid media.
## Here only consider the liquid media, as it is more sensitive to dormant bacteria
prim_eff_res=xo_1021[(xo_1021['XOTEST']=='Primary Efficacy Results') & (xo_1021['XOCAT'].str.contains(analysis_approach,na=False)) &
                            (xo_1021['XOMETHOD'].str.contains('LIQUID'))]
                  
## Add results to dataframe
outcome_tb1021.loc[res_end_of_treat['USUBJID'],'RESULT_AT_END_OF_TREATMENT']=res_end_of_treat.loc[:,'XOSTRESC'].values
outcome_tb1021.loc[res_12_month['USUBJID'],'RESULT_AT_12_MONTHS']=res_12_month.loc[:,'XOSTRESC'].values
outcome_tb1021.loc[prim_eff_res['USUBJID'],'RESULT_LIQUID_MEDIUM_AT_18_MONTHS']=prim_eff_res.loc[:,'XOSTRESC'].values


# EXTRACT TREATMENT FAILURE/RELAPSE/LATE DISCOVERY OF DRUG RESISTANCE OUT OF XO DATASET FOR 1 PHASE III TRIAL (TB-1021)
ds_1021=ds[ds['STUDYID']=='TB-1021']
pat_ids_relapse=ds_1021.loc[ds_1021['DSDECOD'].str.contains('RELAPSE',na=False),'USUBJID'].unique().tolist()
pat_ids_treatment_failure=ds_1021.loc[ds_1021['DSDECOD'].str.contains('FAILURE',na=False),'USUBJID'].unique().tolist()
pat_ids_resistance=ds_1021.loc[ds_1021['DSDECOD'].str.contains('DRUG RESISTANT TB',na=False),'USUBJID'].unique().tolist()

# Add this information to UNFAVOURABLE_OUTCOME_CATEGORY column
outcome_tb1021.loc[outcome_tb1021['RESULT_LIQUID_MEDIUM_AT_18_MONTHS']=='FAVOURABLE','UNFAVOURABLE_OUTCOME_CATEGORY_AT_18_MONTHS'] = 'FAVOURABLE'
outcome_tb1021.loc[pat_ids_relapse, 'UNFAVOURABLE_OUTCOME_CATEGORY_AT_18_MONTHS'] = 'RELAPSE'
outcome_tb1021.loc[pat_ids_treatment_failure, 'UNFAVOURABLE_OUTCOME_CATEGORY_AT_18_MONTHS'] = 'TREATMENT FAILURE'
outcome_tb1021.loc[pat_ids_resistance, 'UNFAVOURABLE_OUTCOME_CATEGORY_AT_18_MONTHS'] = 'LATE DRUG RESISTANCE'

outcome_tb1021['STUDYID']='TB-1021'
outcome_tb1021.to_csv('../data/tb_1021_outcome.csv.gz')

## TB-1020: OUTCOME AT 18 MONTHS
* ### No information on which medium the endpoints were determined on -> extract Eff. Timepoints with Per Protocol approach
* ### Also add reinfection/relapse/early HMR resistance data

In [24]:
xo_1020['XOTEST'].value_counts()
xo_1020['XOCAT'].value_counts()

XOCAT
MODIFIED INTENT TO TREAT    2481
PER PROTOCOL                2481
Name: count, dtype: int64

In [30]:
analysis_approach='PER PROTOCOL'
#analysis_approach='MODIFIED INTENT TO TREAT'

a='Time to Primary Efficacy Endpoint'
a='Primary Efficacy Results'
a='Efficacy Endpoint Details'
a='Primary Efficacy Endpoint Details'

k=[]
for a in ['Primary Efficacy Results','Primary Efficacy Endpoint Details','Time to Primary Efficacy Endpoint']:
    b = xo_1020[(xo_1020['XOTEST']==a)&\
                         (xo_1020['XOCAT'].str.contains(analysis_approach,na=False))][['USUBJID','XOSTRESC']].set_index('USUBJID')
    k.append(b)

c = pd.concat(k,axis=1)
c.columns=['Primary Efficacy Results','Primary Efficacy Endpoint Details','Time to Primary Efficacy Endpoint']
c.groupby('Primary Efficacy Results').apply(lambda x:x['Primary Efficacy Endpoint Details'].value_counts(dropna=False))

/tmp/ipykernel_215460/969535194.py:17: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  c.groupby('Primary Efficacy Results').apply(lambda x:x['Primary Efficacy Endpoint Details'].value_counts(dropna=False))


Primary Efficacy Results  Primary Efficacy Endpoint Details                           
FAVOURABLE                34. Cul Negative at last culture at end fup                     470
UNASSESSABLE              99. Inadequate treatment                                         79
                          03. No culture positive in first two weeks                       55
                          15. Lost to follow-up                                            42
                          02. Inital H/R/M Resistance                                      41
                          11. Culture taken too early                                      34
                          13. Contaminated culture                                         21
                          16. Died from non-TB causes                                      17
                          18. Reinfection                                                   9
                          17. Withdrawn for pregnancy              

In [31]:
c['Time to Primary Efficacy Endpoint']

,Primary Efficacy Results,Primary Efficacy Endpoint Details,Time to Primary Efficacy Endpoint
USUBJID,,,
TB-1020/1003,FAVOURABLE,34. Cul Negative at last culture at end fup,367
TB-1020/1004,FAVOURABLE,34. Cul Negative at last culture at end fup,550
TB-1020/1005,FAVOURABLE,34. Cul Negative at last culture at end fup,457
TB-1020/1006,FAVOURABLE,34. Cul Negative at last culture at end fup,540
TB-1020/1007,FAVOURABLE,34. Cul Negative at last culture at end fup,453
...,...,...,...
TB-1020/5172,UNFAVOURABLE,2. Post-trt; 20. Relapse during follow-up,554
TB-1020/5197,UNFAVOURABLE,2. Post-trt; 20. Relapse during follow-up,196
TB-1020/6010,UNFAVOURABLE,2. Post-trt; 20. Relapse during follow-up,259


In [22]:
c.groupby('Primary Efficacy Results').apply(lambda x:x['Primary Efficacy Endpoint Details'].value_counts(dropna=False))

/tmp/ipykernel_215460/4041308356.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  c.groupby('Primary Efficacy Results').apply(lambda x:x['Primary Efficacy Endpoint Details'].value_counts(dropna=False))


Primary Efficacy Results  Primary Efficacy Endpoint Details                           
FAVOURABLE                34. Cul Negative at last culture at end fup                     470
UNASSESSABLE              99. Inadequate treatment                                         79
                          03. No culture positive in first two weeks                       55
                          15. Lost to follow-up                                            42
                          02. Inital H/R/M Resistance                                      41
                          11. Culture taken too early                                      34
                          13. Contaminated culture                                         21
                          16. Died from non-TB causes                                      17
                          18. Reinfection                                                   9
                          17. Withdrawn for pregnancy              

In [15]:
a='Primary Efficacy Results'
xo_1020[(xo_1020['XOTEST']==a)&\
                     (xo_1020['XOCAT'].str.contains(analysis_approach,na=False))]['XOSTRESC'].value_counts(dropna=False)

XOSTRESC
FAVOURABLE      470
UNASSESSABLE    313
UNFAVOURABLE     44
Name: count, dtype: int64

In [38]:
## Select approach (per protocol vs. modified intent to treat)
analysis_approach='PER PROTOCOL'

## Subset the xo dataframe
xo_1020=xo[xo['STUDYID']=='TB-1020']

## Initialise a dataframe to collect the outcomes to
outcome_columns=['RESULT_AT_18_MONTHS','UNFAVOURABLE_OUTCOME_CATEGORY_AT_18_MONTHS']
outcome_tb1020=pd.DataFrame(index=xo_1020['USUBJID'].unique(),columns=outcome_columns)

## Extract outcomes at 18 months
prim_eff_res=xo_1020[(xo_1020['XOTEST']=='Primary Efficacy Results')&\
                     (xo_1020['XOCAT'].str.contains(analysis_approach,na=False))]

## Add results to dataframe
outcome_tb1020.loc[prim_eff_res['USUBJID'],'RESULT_AT_18_MONTHS']=prim_eff_res.loc[:,'XOSTRESC'].values

# EXTRACT TREATMENT RELAPSE/REINFECTION/INITIAL DISCOVERY OF DRUG RESISTANCE OUT OF XO DATASET 
pat_ids_relapse=xo_1020.loc[(xo_1020['XOCAT'].str.contains('PER',na=False))&\
                            (xo_1020['XOORRES'].str.contains('Relapse',na=False)),'USUBJID'].unique().tolist()
pat_ids_reinfection=xo_1020.loc[(xo_1020['XOCAT'].str.contains('PER',na=False))&\
                            (xo_1020['XOORRES'].str.contains('Reinfection',na=False)),'USUBJID'].unique().tolist()                            
pat_ids_initial_HRM_resistance=xo_1020.loc[(xo_1020['XOCAT'].str.contains('PER',na=False))&\
                            (xo_1020['XOORRES'].str.contains('Resistance',na=False)),'USUBJID'].unique().tolist()  

# Add this information to UNFAVOURABLE_OUTCOME_CATEGORY column
outcome_tb1020.loc[outcome_tb1020['RESULT_AT_18_MONTHS']=='FAVOURABLE','UNFAVOURABLE_OUTCOME_CATEGORY_AT_18_MONTHS'] = 'FAVOURABLE'
outcome_tb1020.loc[pat_ids_relapse, 'UNFAVOURABLE_OUTCOME_CATEGORY_AT_18_MONTHS'] = 'RELAPSE'
outcome_tb1020.loc[pat_ids_reinfection, 'UNFAVOURABLE_OUTCOME_CATEGORY_AT_18_MONTHS'] = 'REINFECTION'
outcome_tb1020.loc[pat_ids_initial_HRM_resistance, 'UNFAVOURABLE_OUTCOME_CATEGORY_AT_18_MONTHS'] = 'INITIAL HRM RESISTANCE'                            

outcome_tb1020['STUDYID']='TB-1020'


## Add time-to-event
time_to_event_df = xo_1020[(xo_1020['XOTEST']=='Time to Primary Efficacy Endpoint')&\
                (xo_1020['XOCAT'].str.contains(analysis_approach,na=False))][['USUBJID','XOSTRESC']].set_index('USUBJID')
outcome_tb1020['TIME_TO_EVENT'] = time_to_event_df.loc[outcome_tb1020.index, 'XOSTRESC'].astype(int).values

outcome_tb1020.to_csv('../data/tb_1020_outcome.csv.gz')

In [ ]:
a='Time to Primary Efficacy Endpoint'
a='Primary Efficacy Results'
a='Efficacy Endpoint Details'
a='Primary Efficacy Endpoint Details'

k=[]
for a in ['Primary Efficacy Results','Primary Efficacy Endpoint Details','Time to Primary Efficacy Endpoint']:
    b = xo_1020[(xo_1020['XOTEST']==a)&\
        (xo_1020['XOCAT'].str.contains(analysis_approach,na=False))][['USUBJID','XOSTRESC']].set_index('USUBJID')

In [39]:
outcome_tb1020

,RESULT_AT_18_MONTHS,UNFAVOURABLE_OUTCOME_CATEGORY_AT_18_MONTHS,STUDYID,TIME_TO_EVENT
TB-1020/1016,UNASSESSABLE,NaN,TB-1020,0
TB-1020/1023,UNASSESSABLE,NaN,TB-1020,0
TB-1020/1029,UNASSESSABLE,NaN,TB-1020,0
TB-1020/1031,UNASSESSABLE,INITIAL HRM RESISTANCE,TB-1020,0
TB-1020/1039,UNASSESSABLE,NaN,TB-1020,0
...,...,...,...,...
TB-1020/6082,FAVOURABLE,FAVOURABLE,TB-1020,808
TB-1020/6084,FAVOURABLE,FAVOURABLE,TB-1020,518
TB-1020/6085,FAVOURABLE,FAVOURABLE,TB-1020,514
TB-1020/6087,FAVOURABLE,FAVOURABLE,TB-1020,505


,RESULT_AT_18_MONTHS,UNFAVOURABLE_OUTCOME_CATEGORY_AT_18_MONTHS,STUDYID
TB-1020/1016,UNASSESSABLE,NaN,TB-1020
TB-1020/1023,UNASSESSABLE,NaN,TB-1020
TB-1020/1029,UNASSESSABLE,NaN,TB-1020
TB-1020/1031,UNASSESSABLE,INITIAL HRM RESISTANCE,TB-1020
TB-1020/1039,UNASSESSABLE,NaN,TB-1020
...,...,...,...
TB-1020/6082,FAVOURABLE,FAVOURABLE,TB-1020
TB-1020/6084,FAVOURABLE,FAVOURABLE,TB-1020
TB-1020/6085,FAVOURABLE,FAVOURABLE,TB-1020
TB-1020/6087,FAVOURABLE,FAVOURABLE,TB-1020


In [34]:
outcome_tb1020['UNFAVOURABLE_OUTCOME_CATEGORY_AT_18_MONTHS'].value_counts(dropna=False)

UNFAVOURABLE_OUTCOME_CATEGORY_AT_18_MONTHS
FAVOURABLE                470
NaN                       271
INITIAL HRM RESISTANCE     41
RELAPSE                    36
REINFECTION                 9
Name: count, dtype: int64

## TB-1022: OUTCOME AT END OF TREATMENT, 18,24 MONTHS
* ### No information on which medium the endpoints were determined on 
* ### Unknown which analysis approach was applied, Modified intention to treat or Per Protocol 
* ### Also add reinfection/relapse/failure information

In [6]:
## Subset the xo dataframe
xo_1022=xo[xo['STUDYID']=='TB-1022']

## Initialise a dataframe to collect the outcomes to
outcome_columns=['RESULT_AT_END_OF_TREATMENT','RESULT_AT_18_MONTHS','RESULT_AT_24_MONTHS','UNFAVOURABLE_OUTCOME_CATEGORY_AT_24_MONTHS']
outcome_tb1022=pd.DataFrame(index=xo_1022['USUBJID'].unique(),columns=outcome_columns)

## Extract outcomes at different timepoints
res_end_of_treat=xo_1022[(xo_1022['XOTEST']=='Efficacy Results')&
                         (xo_1022['XOTPT'].str.contains('END OF TREATMENT'))]   
res_18_month=xo_1022[(xo_1022['XOTEST']=='Efficacy Results')&
                     (xo_1022['XOTPT'].str.contains('18'))]
## For an unknown reason, the  results of 24 months where XOTSTDTL=='TIME TO EVENT' are not matching up with 
#  previous timepoints -> howver, if we consider the datapoints, where XOTSTDTL is NAN. the timepoints match up ->
#  USE THIS!             
prim_eff_res=xo_1022[(xo_1022['XOTEST']=='Efficacy Results')&(xo_1022['XOTSTDTL'].isna())&
                     (xo_1022['XOTPT'].str.contains('24'))]                     

## Add results to dataframe
outcome_tb1022.loc[res_end_of_treat['USUBJID'],'RESULT_AT_END_OF_TREATMENT']=res_end_of_treat.loc[:,'XOSTRESC'].values
outcome_tb1022.loc[res_18_month['USUBJID'],'RESULT_AT_18_MONTHS']=res_18_month.loc[:,'XOSTRESC'].values
outcome_tb1022.loc[prim_eff_res['USUBJID'],'RESULT_AT_24_MONTHS']=prim_eff_res.loc[:,'XOSTRESC'].values


# EXTRACT TREATMENT RELAPSE/REINFECTION/DRUG RESISTANCE OUT OF DS DATASET 
ds_1022=ds[ds['STUDYID']=='TB-1022']
pat_ids_relapse=ds_1022.loc[ds_1022['DSDECOD'].str.contains('RELAPSE',na=False),'USUBJID'].unique().tolist()
pat_ids_treatment_failure=ds_1022.loc[ds_1022['DSDECOD'].str.contains('FAILURE',na=False),'USUBJID'].unique().tolist()
pat_ids_resistance=ds_1022.loc[ds_1022['DSTERM'].str.contains('MDR|RESIST',na=False),'USUBJID'].unique().tolist()
pat_ids_reinfection=ds_1022.loc[ds_1022['DSTERM'].str.contains('REINFECTION',na=False),'USUBJID'].unique().tolist()

# Add this information to UNFAVOURABLE_OUTCOME_CATEGORY column
outcome_tb1022.loc[outcome_tb1022['RESULT_AT_24_MONTHS']=='FAVOURABLE','UNFAVOURABLE_OUTCOME_CATEGORY_AT_24_MONTHS'] = 'FAVOURABLE'
for pat_ids,category in zip([pat_ids_relapse,pat_ids_treatment_failure,pat_ids_resistance,pat_ids_reinfection],\
                             ['RELAPSE','FAILURE','RESISTANCE','REINFECTION']):
    com_ids=list(set(pat_ids)&set(outcome_tb1022.index))
    outcome_tb1022.loc[com_ids, 'UNFAVOURABLE_OUTCOME_CATEGORY_AT_24_MONTHS']=category

## Replace not favourable to unfavourable as this is standard term
outcome_tb1022=outcome_tb1022.replace({'NOT FAVOURABLE':'UNFAVOURABLE'})

outcome_tb1022['STUDYID']='TB-1022'
outcome_tb1022.to_csv('../data/tb_1022_outcome.csv.gz')

## TB-1030 : OUTCOME AT END OF TREATMENT, 12,30 MONTHS INTENTION TO TREAT DATA
* ### No information on which medium the endpoints were determined on
* ### Also add relapse/therapy failure information

In [7]:
## Subset the xo dataframe
xo_1030=xo[xo['STUDYID']=='TB-1030']
print(xo_1030['XOSTRESC'].value_counts(dropna=False))
## Initialise a dataframe to collect the outcomes to
outcome_columns=['RESULT_AT_END_OF_TREATMENT','RESULT_AT_12_MONTHS','RESULT_AT_30_MONTHS','UNFAVOURABLE_OUTCOME_CATEGORY_AT_30_MONTHS']
outcome_tb1030=pd.DataFrame(index=xo_1030['USUBJID'].unique(),columns=outcome_columns)


## Extract outcomes at different timepoints
res_end_of_treat=xo_1030[(xo_1030['XOTEST']=='Status')&
                         (xo_1030['XOTPT'].str.contains('END OF TREATMENT'))]   
res_12_month=xo_1030[(xo_1030['XOTEST']=='Status')&
                     (xo_1030['XOTPT'].str.contains('12'))]

## For an unknown reason, the  results of 24 months where XOTSTDTL=='TIME TO EVENT' are not matching up with 
#  previous timepoints -> howver, if we consider the datapoints, where XOTSTDTL is NAN. the timepoints match up ->
#  USE THIS!             
prim_eff_res=xo_1030[(xo_1030['XOTEST']=='Status')&(xo_1030['XOTPT'].str.contains('30'))] 

## Add results to dataframe
outcome_tb1030.loc[res_end_of_treat['USUBJID'],'RESULT_AT_END_OF_TREATMENT']=res_end_of_treat.loc[:,'XOSTRESC'].values
outcome_tb1030.loc[res_12_month['USUBJID'],'RESULT_AT_12_MONTHS']=res_12_month.loc[:,'XOSTRESC'].values
outcome_tb1030.loc[prim_eff_res['USUBJID'],'RESULT_AT_30_MONTHS']=prim_eff_res.loc[:,'XOSTRESC'].values


# EXTRACT TREATMENT RELAPSE/REINFECTION/INITIAL DISCOVERY OF DRUG RESISTANCE OUT OF XO DATASET
relapse_terms=['5 Relapse within 12m of stopping','5a CH RELP confirmed','7 Relapsed after 12m post Rx',
               '5b CH RELP smear confirm '] 
pat_ids_relapse=xo_1030.loc[(xo_1030['XOORRES'].str.contains('|'.join(relapse_terms),na=False)),'USUBJID'].unique().tolist()

failure_terms=['4a CH FAIL confirmed','4b CH FAIL smear confirm']
pat_ids_treatment_failure=xo_1030.loc[(xo_1030['XOORRES'].str.contains('|'.join(failure_terms),na=False)),'USUBJID'].unique().tolist()


# Add this information to UNFAVOURABLE_OUTCOME_CATEGORY column
outcome_tb1030.loc[outcome_tb1030['RESULT_AT_30_MONTHS']=='FAVORABLE','UNFAVOURABLE_OUTCOME_CATEGORY_AT_30_MONTHS'] = 'FAVOURABLE'
for pat_ids,category in zip([pat_ids_relapse,pat_ids_treatment_failure],['RELAPSE','FAILURE']):                           
    com_ids=list(set(pat_ids)&set(outcome_tb1030.index))
    outcome_tb1030.loc[com_ids, 'UNFAVOURABLE_OUTCOME_CATEGORY_AT_30_MONTHS']=category                                         



## Replace typos and doubtful + not assessed to NaN
outcome_tb1030=outcome_tb1030.replace({'FAVORABLE':'FAVOURABLE','UNFAVORABLE':'UNFAVOURABLE',
                                      'DOUBTFUL':np.nan,'NOT ASSESSED':np.nan})
outcome_tb1030['STUDYID']='TB-1030'
outcome_tb1030.to_csv('../data/tb_1030_outcome.csv.gz')

XOSTRESC
FAVORABLE       2882
NOT ASSESSED     808
UNFAVORABLE      275
DOUBTFUL         100
Name: count, dtype: int64


# TB-1018 OUTCOME AT END OF TREATMENT, 24 MONTHS

In [8]:
ss=pd.read_csv('../../C-Path_data/fullExportDb-1025-Member-CSV/ss.csv',low_memory=False)
xo_1018=ss[(ss['STUDYID']=='TB-1018')&(ss['SSTESTCD']=='TRTSUC')]

## Initialise a dataframe to collect the outcomes to
outcome_columns=['STUDYID','RESULT_AT_END_OF_TREATMENT','RESULT_AT_24_MONTHS']
outcome_tb1018=pd.DataFrame(index=xo_1018['USUBJID'].unique(),columns=outcome_columns)

## Add results to dataframe
outcome_tb1018['STUDYID']='TB-1018'

## Extract outcomes at different timepoints
res_end_of_treat=xo_1018[(xo_1018['VISIT'].str.contains('6',na=False))]   
res_24_month=xo_1018[(xo_1018['VISIT'].str.contains('24',na=False))]  

outcome_tb1018.loc[res_end_of_treat['USUBJID'],'RESULT_AT_END_OF_TREATMENT']=res_end_of_treat.loc[:,'SSSTRESC'].values
outcome_tb1018.loc[res_24_month['USUBJID'],'RESULT_AT_24_MONTHS']=res_24_month.loc[:,'SSSTRESC'].values


In [9]:
outcome_list=[outcome_tb1020,outcome_tb1021,outcome_tb1022,outcome_tb1030,outcome_tb1018]
all=pd.concat(outcome_list,axis=0)
all=all.rename_axis('USUBJID').reset_index()

all=all.replace({'Y':'FAVOURABLE','FAVORABLE':'FAVOURABLE'},regex=True)
all.loc[all['RESULT_AT_18_MONTHS'].isna(),'RESULT_AT_18_MONTHS']=all.loc[all['RESULT_AT_18_MONTHS'].isna(),'RESULT_LIQUID_MEDIUM_AT_18_MONTHS']

all=all.replace({'UNASSESSABLE':np.nan,'NOT ASSESSABLE':np.nan,'NOT ASSESSED':np.nan,})

all=all.dropna(how='all',axis=0)
all.to_csv('../data/tb_1018_20_21_22_30_outcome.csv.gz',compression='gzip')
